## §1 Setup & Imports

This notebook presents and discusses the final results of the CICIoT2023
multi-class IoT intrusion detection project. No model training occurs here —
all metrics are loaded from files saved by notebook 03. Results are organised
around the central evaluation metric (macro F1) and discussed in the context
of the state of the art.

In [ ]:
import json, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

PROCESSED_DIR = '../data/processed/'
CLASS_NAMES   = ['Benign','BruteForce','DDoS','DoS','Mirai','Recon','Spoofing','Web-based']

df_val  = pd.read_csv(PROCESSED_DIR + 'results_val.csv').set_index('Model')
df_test = pd.read_csv(PROCESSED_DIR + 'results_test.csv').set_index('Model')
best_hparams   = json.load(open(PROCESSED_DIR + 'best_hparams.json'))
best_names     = json.load(open(PROCESSED_DIR + 'best_model_name.json'))
best_val_name  = best_names['val_best']
best_test_name = best_names['test_best']

print(f'Models evaluated : {list(df_test.index)}')
print(f'Best (val)  : {best_val_name}')
print(f'Best (test) : {best_test_name}')

## §2 Validation vs Test Results

Comparing validation and test macro F1 detects overfitting to the
validation set during Optuna's 50-trial search. A negative delta means
the model performs worse on unseen data than on the set used for
hyperparameter selection. A delta close to zero indicates robust
generalisation. Large negative deltas would suggest the number of Optuna
trials should be reduced or a held-out pruning set introduced.

In [ ]:
delta = (df_test['Macro F1'] - df_val['Macro F1']).rename('Delta F1')
comparison = pd.concat([
    df_val['Macro F1'].rename('Val F1'),
    df_test['Macro F1'].rename('Test F1'),
    delta
], axis=1).sort_values('Test F1', ascending=False)
print(comparison.to_string(float_format='{:.4f}'.format))
print()
print(f'Max overfit (most negative delta): '
      f'{comparison["Delta F1"].idxmin()} ({comparison["Delta F1"].min():.4f})')

## §3 Main Results Table & Ranking

The dashed reference line at F1=0.70 corresponds to the best result
reported in the original CICIoT2023 paper (Neto et al., 2023), achieved
with Random Forest on 8 classes without imbalance handling. All models
in this project apply RandomOverSampler, which substantially improves
recall on minority classes and raises the macro F1 floor. The gap between
the baseline and our results quantifies the benefit of imbalance mitigation
on this dataset.

In [ ]:
df_ranked = df_test.sort_values('Macro F1', ascending=False)
print('=== Test Set Results — Ranked by Macro F1 ===')
print(df_ranked.to_string(float_format='{:.4f}'.format))
print()

NETO_F1 = 0.70

colors = {
    'Decision Tree':     '#aaaaaa',
    'Random Forest':     '#4878d0',
    'AdaBoost':          '#ee854a',
    'Gradient Boosting': '#d65f5f',
    'XGBoost':           '#e8851a',
    'Voting (hard)':     '#6acc65',
    'Voting (soft)':     '#56ae6c',
    'Stacking':          '#956cb4',
}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    df_ranked.index,
    df_ranked['Macro F1'],
    color=[colors.get(m, '#888888') for m in df_ranked.index]
)
ax.axvline(NETO_F1, color='red', linestyle='--', linewidth=1.5,
           label=f'Neto et al. (2023) RF baseline — F1={NETO_F1}')
for bar in bars:
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.4f}', va='center', fontsize=9)
ax.set_xlim(0, 1.08)
ax.set_xlabel('Macro F1 (test set)')
ax.set_title('Model Comparison — Macro F1 on Test Set')
ax.legend()
plt.tight_layout()
plt.show()

## §4 Accuracy vs Macro F1 — The Imbalance Trap

This is the central argument for macro F1 over accuracy in imbalanced settings.
With DDoS comprising 75.6% of the test set, a degenerate classifier predicting
only DDoS achieves 75.6% accuracy with zero macro F1. The accuracy-F1 gap
quantifies how much accuracy flatters each model: large gaps indicate the model
is performing well on majority classes while failing on BruteForce (~0.02% of test)
and Web-based (~0.05% of test). MCC is the most conservative metric as it
penalises failure on any class regardless of size.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, metric in zip(axes, ['Accuracy', 'Macro F1']):
    vals = df_test[metric].sort_values(ascending=True)
    ax.barh(vals.index, vals.values,
            color=['#4878d0' if m != 'Decision Tree' else '#aaaaaa'
                   for m in vals.index])
    ax.set_xlim(0.5, 1.02)
    ax.set_title(metric)
    ax.set_xlabel('Score')
    for i, v in enumerate(vals.values):
        ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

axes[0].set_ylabel('Model')
plt.suptitle('Accuracy vs Macro F1 — Imbalance Effect', fontsize=13)
plt.tight_layout()
plt.show()

gap = (df_test['Accuracy'] - df_test['Macro F1']).sort_values(ascending=False)
print('Accuracy − Macro F1 gap (larger = more misleading accuracy):')
print(gap.to_string(float_format='{:.4f}'.format))

## §5 Full Metric Heatmap

The heatmap provides a compact view of all trade-offs simultaneously.
Rows are sorted by macro F1. Models with high macro P but lower macro R
(or vice versa) reveal a precision-recall trade-off: RandomOverSampler
boosts recall on minority classes at some cost to precision, since
duplicate minority samples can blur the decision boundary.

In [ ]:
metrics = ['Accuracy','Macro F1','Macro P','Macro R','MCC']
heat_data = df_test[metrics].sort_values('Macro F1', ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(
    heat_data, annot=True, fmt='.4f', cmap='YlGn',
    vmin=0.5, vmax=1.0, linewidths=0.5,
    cbar_kws={'label': 'Score'}, ax=ax
)
ax.set_title('All Models × All Metrics — Test Set')
ax.set_xlabel('')
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

## §6 Hyperparameter Analysis

The optimal hyperparameters reflect the structure of the problem.
High n_estimators with moderate max_depth in RF suggests the problem
benefits from ensemble diversity rather than deep individual trees.
Low learning_rate paired with high n_estimators in XGBoost and GBM
indicates a smooth loss landscape — cautious steps generalise better
than aggressive ones on this imbalanced multi-class problem. XGBoost's
L1/L2 regularisation values (reg_alpha, reg_lambda) quantify how much
sparsity and weight decay were needed to prevent overfitting on the
oversampled training set.

In [ ]:
rows = []
for model_name, hparams in best_hparams.items():
    for param, value in hparams.items():
        rows.append({'Model': model_name, 'Hyperparameter': param, 'Value': value})
df_hp = pd.DataFrame(rows)
print(df_hp.to_string(index=False))

## §7 Comparison with State of the Art

Direct comparison with the literature requires caution — studies differ in
train/test splits, feature sets, and preprocessing pipelines. The two
studies achieving >98% macro F1 (Al-Ibaisi et al., Anis 2024) both apply
explicit feature selection before training: Random Forest Regressor importance
ranking and Chi-square tests respectively. This project does not apply feature
selection beyond dropping constant columns and one highly correlated pair
identified in the EDA. The gap between our results and the state of the art
is therefore primarily attributable to feature selection, not model choice —
an actionable finding for future work.

In [ ]:
best_acc = df_test.loc[best_test_name, 'Accuracy']
best_f1  = df_test.loc[best_test_name, 'Macro F1']

sota = pd.DataFrame([
    {'Study': 'Neto et al. (2023)',       'Model': 'Random Forest',
     'Task': '8-class', 'Accuracy': '>0.98', 'Macro F1': '~0.70'},
    {'Study': 'Al-Ibaisi et al. (2025)',  'Model': 'DT + RFR feat. sel.',
     'Task': 'Multi-class', 'Accuracy': '0.9999', 'Macro F1': '~0.99'},
    {'Study': 'Anis (2024)',              'Model': 'XGBoost + RF feat. sel.',
     'Task': '8-class', 'Accuracy': '>0.99', 'Macro F1': '>0.98'},
    {'Study': 'This project',             'Model': best_test_name,
     'Task': '8-class',
     'Accuracy': f'{best_acc:.4f}',
     'Macro F1': f'{best_f1:.4f}'},
])
print(sota.to_string(index=False))

## §8 Lessons Learned

**1. SMOTE underperformed RandomOverSampler.** With BruteForce at 118 train
samples, SMOTE interpolates between very few neighbours in a 39-dimensional
space. The synthetic samples are geometrically close to real ones but lie
in regions of the feature space that may not correspond to real attack patterns,
introducing noise that confuses tree-based models. RandomOverSampler's exact
duplicates are less informative but less harmful — a case where simplicity wins.

**2. Accuracy is a dangerous metric for imbalanced multiclass classification.**
The Decision Tree baseline achieves ~99% accuracy while producing near-zero
recall on BruteForce and Web-based. Reporting only accuracy would lead to the
false conclusion that the problem is essentially solved. Macro F1 and MCC
expose this failure by weighting all classes equally regardless of frequency.

**3. GradientBoostingClassifier's lack of intra-tree parallelism is a practical
bottleneck at scale.** Unlike RF (n_jobs=-1 per tree) and XGBoost (GPU),
sklearn GBM scales as O(n_estimators × n_samples). At 4.2M resampled rows
it becomes impractical; the 500K subsample is a justified engineering trade-off.
XGBoost with CUDA solved the same problem more elegantly.

**4. Optuna with MedianPruner significantly accelerates HPT.** The pruner stops
trials that are unlikely to beat the current best after observing early
intermediate values — concentrating compute on promising hyperparameter
regions. Combined with the X_opt subsample (100K), 50 trials complete in
minutes rather than hours.

**5. The stratified sampling pipeline is methodologically superior to ad-hoc approaches.**
The compute_sample_quota.py → notebooks 01–04 pipeline derives proportions from a
full label-column scan and makes the sampling decision explicit, auditable, and
reproducible without hardcoding anything in the notebooks.

**6. BruteForce (169 total samples after stratified sampling) is the hardest class.**
With ~118 training samples, no resampling technique can compensate for the absence
of real diversity in the feature space. Collecting more genuine BruteForce traffic
samples is the single highest-impact improvement available — no algorithmic
improvement can substitute for data.

**7. Feature selection was not applied beyond EDA findings.**
Literature shows that applying RFR importance ranking or Chi-square selection
before training pushes macro F1 above 98% on this same dataset. The EDA identified
candidates (|r|>0.95 pairs); acting on them is the clearest path to closing the gap
with the state of the art.

## §9 Conclusions

The best-performing model on the test set is determined in notebook 03 §12.
All ensemble methods substantially outperform the Neto et al. (2023) baseline
(macro F1 ~0.70) — the improvement quantifies the direct benefit of applying
RandomOverSampler before training on this heavily imbalanced dataset.

RandomOverSampler outperformed SMOTE and ADASYN on this dataset. The reason is
structural: with only 118 BruteForce training samples in a 39-dimensional feature
space, SMOTE's synthetic interpolation produces geometrically valid but semantically
questionable samples. Exact duplication (RandomOverSampler) is less informative but
avoids introducing noisy out-of-distribution points near the decision boundary.

The full pipeline is reproducible end-to-end:
`compute_sample_quota.py → 01_data_loading_sampling → 02_imbalance_handling → 03_model_comparison → 04_results_discussion`.
Each notebook loads its inputs from `../data/processed/` and saves its outputs there,
with no hardcoded paths or data transformations shared across files.

**Three concrete next steps ranked by expected impact:**

1. **Feature selection** (RFR importance ranking + correlation pruning from the EDA
   |r|>0.95 candidate list) — literature results on this exact dataset suggest a
   macro F1 above 0.95 is achievable. This is the clearest gap between this project
   and the state of the art.

2. **Additional BruteForce data collection** — with only 169 total samples (118 train),
   this class is the binding constraint on minority-class recall. No algorithmic
   improvement can substitute for greater data diversity in the feature space.

3. **Deep learning (LSTM or Transformer)** — capturing temporal dependencies in
   network flow sequences could improve classification of attack types with similar
   statistical features, particularly the Web-based ↔ Benign and BruteForce ↔ Recon
   confusions that persist across all models (Chen et al., 2024).